In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [23]:
corpus = """
The old clock on the mantle ticked slowly, each swing a steady reminder of time passing.
Dust motes danced in the sunlight streaming through the windowpanes, illuminating the quiet room.
A cat, curled on a velvet cushion, twitched its tail in a dream.
Outside, the wind whispered through the leaves of ancient trees,
carrying the scent of damp earth and distant rain.
A book lay open on the table, its pages filled with stories of faraway lands and brave heroes.
The air was still, thick with the unspoken memories held within the walls.
It was a moment of peace, a pause in the hurried rhythm of the world,
where time seemed to stretch and soften, allowing the soul to catch its breath.
The world outside continued its endless turning, but here, in this small corner,
a quiet magic held sway.
"""

In [24]:
# Preprocessing
corpus = corpus.lower().replace("\n", " ")
tokenizer = Tokenizer(char_level=True)  # character-level
tokenizer.fit_on_texts([corpus])
total_chars = len(tokenizer.word_index) + 1

In [25]:
# Convert text to sequence of integers
encoded = tokenizer.texts_to_sequences([corpus])[0]

In [26]:
# Prepare input-output sequences
seq_length = 40
sequences = []
for i in range(seq_length, len(encoded)):
    seq = encoded[i-seq_length:i]
    label = encoded[i]
    sequences.append((seq, label))

In [27]:
X = np.array([s[0] for s in sequences])
y = to_categorical([s[1] for s in sequences], num_classes=total_chars)

In [28]:
# Build LSTM model
model = Sequential([
    Embedding(total_chars, 50, input_length=seq_length),
    LSTM(128, return_sequences=False),
    Dense(total_chars, activation='softmax')
])

In [29]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [30]:
model.fit(X, y, epochs=40, verbose=1)

Epoch 1/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.1448 - loss: 3.2037
Epoch 2/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1748 - loss: 2.9120
Epoch 3/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1808 - loss: 2.8843
Epoch 4/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1656 - loss: 2.8717
Epoch 5/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1851 - loss: 2.8343
Epoch 6/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1956 - loss: 2.7624
Epoch 7/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2242 - loss: 2.6910
Epoch 8/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2377 - loss: 2.6635
Epoch 9/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3004 - loss: 2.5400
Epoch 10/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2975 - loss: 2.4976
Epoch 11/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3108 - loss: 2.4112
Epoch 12/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3207 - lo

In [31]:
# Function to generate text
def generate_text(model, tokenizer, seq_length, seed_text, n_chars):
    result = []
    in_text = seed_text.lower()
    for _ in range(n_chars):
        encoded = tokenizer.texts_to_sequences([in_text])[-1]
        encoded = encoded[-seq_length:]  # last seq_length chars
        encoded = tf.keras.preprocessing.sequence.pad_sequences([encoded], maxlen=seq_length, padding='pre')
        y_pred = np.argmax(model.predict(encoded, verbose=0))
        out_char = ''
        for char, index in tokenizer.word_index.items():
            if index == y_pred:
                out_char = char
                break
        in_text += out_char
        result.append(out_char)
    return seed_text + ''.join(result)

In [33]:
# Generate text
print(generate_text(model, tokenizer, seq_length, "The old clock", 100))

The old clock on the and in the surled of the world outs meale,  a quiet of aced ta sand in the surled of the wor
